<a href="https://colab.research.google.com/github/erizov/Aphorium/blob/main/%D0%92%D0%B5%D0%B1%D0%B8%D0%BD%D0%B0%D1%80_RAG_1_%D0%B0%D0%BF%D1%80%D0%B5%D0%BB%D1%8F_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# RAG BACKEND MULTI-STEP (FASTAPI + CLOUDFLARE, POLLING FINAL ANSWER) FIXED
# ============================================

# ---------- 0) Clean old processes ----------
!pkill -f uvicorn || true
!pkill -f cloudflared || true
!rm -f /usr/local/bin/cloudflared || true

# ---------- 1) Install ----------
!pip -q install fastapi "uvicorn[standard]" numpy openai tiktoken
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import time
time.sleep(2)

# ---------- 2) Inject API key from Colab Secrets ----------
from google.colab import userdata
import os

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception as e:
    print("⚠️ Failed to load API key:", e)

# Optional:
# os.environ["OPENAI_MODEL"] = "gpt-4o-mini"

# ---------- 3) Backend code ----------
backend_code = r'''
import os
import re
import json
import time
import uuid
import socket
import asyncio
import threading
from typing import Dict, Any, List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

import tiktoken

# ===== OpenAI init =====
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found in environment")

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# =========================
# CONFIG
# =========================
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USD_TO_RUB = 80
PRICE_INPUT = 0.00001
PRICE_OUTPUT = 0.00003
DEFAULT_TOP_K = 5
DEFAULT_MULTI_QUERY_COUNT = 5
DEFAULT_RERANK_TOP_N = 5
MAX_CONTEXT_CHARS_FOR_RERANK = 3500
MAX_CHUNK_CHARS_FOR_RERANK = 1600

app = FastAPI(title="RAG Multi-Step Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# =========================
# STORES
# =========================
sessions_store: Dict[str, Dict[str, Any]] = {}
requests_store: Dict[str, Dict[str, Any]] = {}

# =========================
# UTILS
# =========================
def chunk_text(text: str, chunk_size: int = 1000) -> List[str]:
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


def simple_search_with_scores(query: str, chunks: List[str], top_k: int = 5) -> List[Dict[str, Any]]:
    query_words = [w.strip().lower() for w in re.findall(r"\w+", query, flags=re.UNICODE) if w.strip()]
    scored = []

    for idx, c in enumerate(chunks):
        c_lower = c.lower()
        exact_hits = sum(word in c_lower for word in query_words)
        unique_hits = len(set(word for word in query_words if word in c_lower))
        char_overlap_bonus = min(len(query) / max(len(c), 1), 1.0)
        score = float(exact_hits * 1.0 + unique_hits * 0.5 + char_overlap_bonus * 0.2)

        scored.append({
            "global_chunk_index": idx,
            "text": c,
            "retrieval_score": round(score, 4)
        })

    ranked = sorted(scored, key=lambda x: x["retrieval_score"], reverse=True)
    return ranked[:top_k]


def count_tokens(text: str) -> int:
    enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))


def now_ts() -> float:
    return time.time()


def safe_json_loads(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        return json.loads(m.group(0))

    raise ValueError("Model did not return valid JSON")


def llm_json(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.2,
    max_tokens: int = 1200
) -> Dict[str, Any]:
    started = now_ts()

    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    content = resp.choices[0].message.content or "{}"
    data = safe_json_loads(content)

    prompt_tokens = getattr(resp.usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(resp.usage, "completion_tokens", 0) or 0
    total_tokens = getattr(resp.usage, "total_tokens", prompt_tokens + completion_tokens)

    return {
        "data": data,
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens
        },
        "cost": {
            "usd": prompt_tokens * PRICE_INPUT + completion_tokens * PRICE_OUTPUT,
            "rub": (prompt_tokens * PRICE_INPUT + completion_tokens * PRICE_OUTPUT) * USD_TO_RUB
        },
        "timing": {
            "total_time_sec": round(now_ts() - started, 2)
        }
    }


def llm_text(
    system_prompt: str,
    user_prompt: str,
) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


def build_usage_fallback(prompt_text: str, answer_text: str) -> Dict[str, Any]:
    pt = count_tokens(prompt_text)
    ct = count_tokens(answer_text)
    cost = pt * PRICE_INPUT + ct * PRICE_OUTPUT
    return {
        "usage": {
            "prompt_tokens": pt,
            "completion_tokens": ct,
            "total_tokens": pt + ct
        },
        "cost": {
            "usd": cost,
            "rub": cost * USD_TO_RUB
        }
    }


def new_session(user_id: Optional[str] = None) -> Dict[str, Any]:
    session_id = str(uuid.uuid4())
    session = {
        "session_id": session_id,
        "user_id": user_id,
        "status": "created",
        "created_at": now_ts(),
        "updated_at": now_ts(),

        "original_question": None,
        "clarifying_question": None,
        "clarifying_answer": None,
        "final_user_query": None,

        "multi_queries": [],
        "retrieval_results": [],
        "retrieval_flat_chunks": [],

        "reranked_chunks": [],
        "top_chunks": [],

        "final_answer": None,
        "final_answer_stream_request_id": None,

        "metrics": {
            "steps": {}
        }
    }
    sessions_store[session_id] = session
    return session


def get_session_or_404(session_id: str) -> Dict[str, Any]:
    session = sessions_store.get(session_id)
    if not session:
        raise HTTPException(status_code=404, detail="session_id not found")
    return session


def update_step_metrics(session: Dict[str, Any], step_name: str, payload: Dict[str, Any]):
    session["metrics"]["steps"][step_name] = payload
    session["updated_at"] = now_ts()


def compose_final_query(original_question: str, clarifying_question: str, clarifying_answer: str) -> Dict[str, Any]:
    system_prompt = """
Ты помогаешь уточнить пользовательский запрос для RAG-поиска.
Верни JSON.
""".strip()

    user_prompt = f"""
Собери финальный уточнённый запрос для поиска по базе знаний.

Исходный вопрос:
{original_question}

Уточняющий вопрос:
{clarifying_question}

Ответ пользователя на уточнение:
{clarifying_answer}

Верни JSON строго такого вида:
{{
  "final_user_query": "..."
}}
""".strip()

    result = llm_json(system_prompt, user_prompt, temperature=0.1, max_tokens=300)
    return result


def generate_clarifying_question(original_question: str) -> Dict[str, Any]:
    system_prompt = """
Ты AI-ассистент для RAG по базе знаний.
Твоя задача — задать РОВНО ОДИН уточняющий вопрос к исходному запросу пользователя.
Если вопрос уже достаточно конкретен, всё равно задай одно краткое уточнение, которое улучшит поиск по базе знаний.
Верни JSON.
""".strip()

    user_prompt = f"""
Пользовательский вопрос:
{original_question}

Верни JSON строго такого вида:
{{
  "clarifying_question": "..."
}}
""".strip()

    result = llm_json(system_prompt, user_prompt, temperature=0.2, max_tokens=250)
    return result


def generate_multi_queries(final_user_query: str, count: int = 5) -> Dict[str, Any]:
    system_prompt = """
Ты генерируешь несколько близких, но немного отличающихся поисковых формулировок для RAG.
Они должны помогать найти разные релевантные фрагменты знаний по базе.
Верни JSON.
""".strip()

    user_prompt = f"""
Финальный уточнённый запрос:
{final_user_query}

Сгенерируй {count} поисковых запросов:
- они должны быть близки по смыслу
- но немного различаться по формулировке и акцентам
- они не должны быть слишком длинными
- они должны подходить для поиска по базе знаний

Верни JSON строго такого вида:
{{
  "multi_queries": [
    "...",
    "...",
    "...",
    "...",
    "..."
  ]
}}
""".strip()

    result = llm_json(system_prompt, user_prompt, temperature=0.3, max_tokens=700)
    return result


def flatten_retrieval_results(retrieval_results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    flat = []
    for item in retrieval_results:
        query_id = item["query_id"]
        query = item["query"]
        for c in item["chunks"]:
            flat.append({
                "chunk_id": c["chunk_id"],
                "query_id": query_id,
                "query": query,
                "text": c["text"],
                "retrieval_score": c["retrieval_score"],
                "global_chunk_index": c.get("global_chunk_index")
            })
    return flat


def build_rerank_payload(final_user_query: str, flat_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    compact_chunks = []
    for c in flat_chunks:
        compact_chunks.append({
            "chunk_id": c["chunk_id"],
            "query_id": c["query_id"],
            "query": c["query"][:300],
            "retrieval_score": c["retrieval_score"],
            "text": c["text"][:MAX_CHUNK_CHARS_FOR_RERANK]
        })

    system_prompt = """
Ты делаешь rerank чанков для RAG по базе знаний.
Оцени каждый чанк относительно исходного финального запроса пользователя.
Шкала: от 1 до 10, где 10 = максимально полезный для финального ответа.
Для каждого чанка дай короткий комментарий.
После этого выбери top 5 самых полезных чанков.

Верни только JSON.
""".strip()

    user_prompt = f"""
Финальный запрос пользователя:
{final_user_query[:MAX_CONTEXT_CHARS_FOR_RERANK]}

Вот чанки для оценки:
{json.dumps(compact_chunks, ensure_ascii=False)}

Верни JSON строго такого вида:
{{
  "reranked_chunks": [
    {{
      "chunk_id": "...",
      "rerank_score_10": 8,
      "comment": "..."
    }}
  ],
  "top_chunk_ids": ["...", "...", "...", "...", "..."]
}}
""".strip()

    result = llm_json(system_prompt, user_prompt, temperature=0.1, max_tokens=2500)
    return result


def build_final_answer_prompt(final_user_query: str, top_chunks: List[Dict[str, Any]]) -> str:
    chunks_text = []
    for idx, ch in enumerate(top_chunks, start=1):
        chunks_text.append(
            f"[Chunk {idx} | chunk_id={ch['chunk_id']} | rerank={ch.get('rerank_score_10')}]\\n{ch['text']}"
        )

    joined_chunks = "\\n\\n".join(chunks_text)

    prompt = f"""
Ты AI-консультант, отвечающий строго по предоставленным чанкам базы знаний.

КРИТИЧЕСКИ ВАЖНО:
1. Ниже уже переданы релевантные чанки, отобранные по запросу пользователя.
2. Если в чанках есть хоть какая-то полезная информация по вопросу, НЕЛЬЗЯ писать, что информации нет.
3. Нужно извлечь максимум пользы из этих чанков и дать практичный ответ.
4. Фразу о недостатке информации используй только если в чанках реально вообще нет относящихся к вопросу сведений.
5. Не придумывай факты вне чанков, но пересказывай и структурируй информацию из чанков нормально и уверенно.

Финальный вопрос пользователя:
{final_user_query}

Топ-чанки:
{joined_chunks}

Сформируй понятный, аккуратный, практичный ответ строго по этим чанкам.
Если данные в чанках обрывочные, всё равно дай лучший возможный ответ на их основе и отдельно коротко укажи, чего именно не хватает.
""".strip()

    return prompt


def ensure_retrieval(session: Dict[str, Any], top_k_per_query: int = DEFAULT_TOP_K):
    if session["retrieval_results"]:
        return

    if not session["multi_queries"]:
        raise HTTPException(status_code=400, detail="multi_queries not generated yet")

    start_time = now_ts()
    retrieval_results = []

    for q_idx, query in enumerate(session["multi_queries"], start=1):
        found = simple_search_with_scores(query, RAG_CHUNKS, top_k=top_k_per_query)
        chunks_payload = []

        for c_idx, item in enumerate(found, start=1):
            chunks_payload.append({
                "chunk_id": f"q{q_idx}_c{c_idx}",
                "global_chunk_index": item["global_chunk_index"],
                "text": item["text"],
                "retrieval_score": item["retrieval_score"]
            })

        retrieval_results.append({
            "query_id": q_idx,
            "query": query,
            "chunks": chunks_payload
        })

    session["retrieval_results"] = retrieval_results
    session["retrieval_flat_chunks"] = flatten_retrieval_results(retrieval_results)
    session["status"] = "retrieved"

    update_step_metrics(session, "retrieve", {
        "timing": {
            "total_time_sec": round(now_ts() - start_time, 2)
        },
        "meta": {
            "total_queries": len(session["multi_queries"]),
            "top_k_per_query": top_k_per_query,
            "total_chunks": len(session["retrieval_flat_chunks"])
        }
    })


def ensure_rerank(session: Dict[str, Any], top_n: int = DEFAULT_RERANK_TOP_N):
    if session["top_chunks"]:
        return

    if not session["retrieval_flat_chunks"]:
        raise HTTPException(status_code=400, detail="retrieval not done yet")

    result = build_rerank_payload(session["final_user_query"], session["retrieval_flat_chunks"])
    data = result["data"]

    reranked_map = {
        item["chunk_id"]: item for item in data.get("reranked_chunks", [])
    }
    top_ids = data.get("top_chunk_ids", [])[:top_n]

    merged = []
    for c in session["retrieval_flat_chunks"]:
        rr = reranked_map.get(c["chunk_id"], {})
        merged.append({
            **c,
            "rerank_score_10": int(rr.get("rerank_score_10", 1)),
            "comment": rr.get("comment", ""),
            "selected": c["chunk_id"] in top_ids
        })

    merged_sorted = sorted(
        merged,
        key=lambda x: (x["selected"], x["rerank_score_10"], x["retrieval_score"]),
        reverse=True
    )

    primary_top_chunks = [
        x for x in merged_sorted
        if x["chunk_id"] in top_ids and int(x.get("rerank_score_10", 1)) >= 5
    ]

    if len(primary_top_chunks) < top_n:
        fallback_extra = [
            x for x in merged_sorted
            if x["chunk_id"] not in {t["chunk_id"] for t in primary_top_chunks}
        ]
        primary_top_chunks.extend(fallback_extra[:max(0, top_n - len(primary_top_chunks))])

    top_chunks = sorted(
        primary_top_chunks,
        key=lambda x: (x["rerank_score_10"], x["retrieval_score"]),
        reverse=True
    )[:top_n]

    session["reranked_chunks"] = merged_sorted
    session["top_chunks"] = top_chunks
    session["status"] = "reranked"

    update_step_metrics(session, "rerank", {
        "usage": result["usage"],
        "cost": result["cost"],
        "timing": result["timing"],
        "meta": {
            "input_chunks": len(session["retrieval_flat_chunks"]),
            "selected_top_n": len(session["top_chunks"])
        }
    })


def start_final_answer_job(session: Dict[str, Any]) -> str:
    if not session["top_chunks"]:
        raise HTTPException(status_code=400, detail="top_chunks not prepared yet")

    request_id = str(uuid.uuid4())
    session["final_answer_stream_request_id"] = request_id

    requests_store[request_id] = {
        "status": "processing",
        "result": None,
        "started_at": now_ts()
    }

    system_prompt = (
        "Ты AI-консультант по базе знаний. "
        "Отвечай только на основе переданных чанков. "
        "Если чанки релевантны, обязательно извлеки из них полезный ответ. "
        "Не говори, что информации нет, если в чанках упоминаются нужные шаги, параметры, инструкции, бюджет, этапы или примеры."
    )

    user_prompt = build_final_answer_prompt(session["final_user_query"], session["top_chunks"])

    def process():
        try:
            start_time = now_ts()
            answer = llm_text(system_prompt, user_prompt)

            usage_cost = build_usage_fallback(system_prompt + "\\n\\n" + user_prompt, answer)

            result = {
                "session_id": session["session_id"],
                "final_user_query": session["final_user_query"],
                "used_chunk_ids": [c["chunk_id"] for c in session["top_chunks"]],
                "used_chunks_count": len(session["top_chunks"]),
                "top_chunks": session["top_chunks"],
                "final_answer": answer,
                "usage": usage_cost["usage"],
                "cost": usage_cost["cost"],
                "timing": {
                    "total_time_sec": round(now_ts() - start_time, 2)
                },
                "source": "polling"
            }

            requests_store[request_id]["result"] = result
            requests_store[request_id]["status"] = "done"

            session["final_answer"] = answer
            session["status"] = "answered"

            update_step_metrics(session, "answer_generate", {
                "usage": result["usage"],
                "cost": result["cost"],
                "timing": result["timing"],
                "meta": {
                    "used_chunk_ids": result["used_chunk_ids"],
                    "used_chunks_count": result["used_chunks_count"]
                }
            })

        except Exception as e:
            requests_store[request_id]["status"] = "error"
            requests_store[request_id]["result"] = {"error": str(e)}

    threading.Thread(target=process, daemon=True).start()

    return request_id


# =========================
# LOAD RAG
# =========================
RAG_PATH = "/content/RAG.txt"

if os.path.exists(RAG_PATH):
    with open(RAG_PATH, "r", encoding="utf-8") as f:
        RAG_TEXT = f.read()
else:
    RAG_TEXT = "RAG.txt not found"

RAG_CHUNKS = chunk_text(RAG_TEXT, chunk_size=1000)

# =========================
# ROUTES
# =========================

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL,
        "rag_chunks": len(RAG_CHUNKS)
    }


@app.post("/session/start")
def session_start(data: Dict[str, Any]):
    user_id = data.get("user_id")
    session = new_session(user_id=user_id)
    return {
        "session_id": session["session_id"],
        "status": "created"
    }


@app.get("/session/{session_id}")
def session_get(session_id: str):
    session = get_session_or_404(session_id)
    return session


@app.post("/question/clarify")
def question_clarify(data: Dict[str, Any]):
    session_id = data.get("session_id")
    question = (data.get("question") or "").strip()

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")
    if not question:
        raise HTTPException(status_code=400, detail="question is required")

    session = get_session_or_404(session_id)

    result = generate_clarifying_question(question)
    clarify_data = result["data"]
    clarifying_question = (clarify_data.get("clarifying_question") or "").strip()

    session["original_question"] = question
    session["clarifying_question"] = clarifying_question
    session["status"] = "clarifying_question_generated"

    update_step_metrics(session, "question_clarify", {
        "usage": result["usage"],
        "cost": result["cost"],
        "timing": result["timing"]
    })

    return {
        "session_id": session_id,
        "original_question": question,
        "clarifying_question": clarifying_question
    }


@app.post("/question/clarify/answer")
def question_clarify_answer(data: Dict[str, Any]):
    session_id = data.get("session_id")
    clarifying_answer = (data.get("clarifying_answer") or "").strip()

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")
    if not clarifying_answer:
        raise HTTPException(status_code=400, detail="clarifying_answer is required")

    session = get_session_or_404(session_id)

    if not session["original_question"] or not session["clarifying_question"]:
        raise HTTPException(status_code=400, detail="clarifying question not generated yet")

    session["clarifying_answer"] = clarifying_answer

    result = compose_final_query(
        original_question=session["original_question"],
        clarifying_question=session["clarifying_question"],
        clarifying_answer=clarifying_answer
    )

    final_user_query = (result["data"].get("final_user_query") or "").strip()
    session["final_user_query"] = final_user_query
    session["status"] = "clarified"

    update_step_metrics(session, "question_clarify_answer", {
        "usage": result["usage"],
        "cost": result["cost"],
        "timing": result["timing"]
    })

    return {
        "session_id": session_id,
        "original_question": session["original_question"],
        "clarifying_question": session["clarifying_question"],
        "clarifying_answer": session["clarifying_answer"],
        "final_user_query": session["final_user_query"]
    }


@app.post("/query/multi")
def query_multi(data: Dict[str, Any]):
    session_id = data.get("session_id")
    count = int(data.get("count", DEFAULT_MULTI_QUERY_COUNT))

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")

    session = get_session_or_404(session_id)

    if not session["final_user_query"]:
        raise HTTPException(status_code=400, detail="final_user_query not prepared yet")

    result = generate_multi_queries(session["final_user_query"], count=count)
    multi_queries = result["data"].get("multi_queries", [])

    if not isinstance(multi_queries, list):
        raise HTTPException(status_code=500, detail="multi_queries invalid format")

    multi_queries = [str(x).strip() for x in multi_queries if str(x).strip()][:count]

    session["multi_queries"] = multi_queries
    session["status"] = "multi_query_generated"

    update_step_metrics(session, "query_multi", {
        "usage": result["usage"],
        "cost": result["cost"],
        "timing": result["timing"],
        "meta": {
            "generated_queries": len(multi_queries)
        }
    })

    return {
        "session_id": session_id,
        "final_user_query": session["final_user_query"],
        "multi_queries": session["multi_queries"]
    }


@app.post("/retrieve")
def retrieve(data: Dict[str, Any]):
    session_id = data.get("session_id")
    top_k_per_query = int(data.get("top_k_per_query", DEFAULT_TOP_K))

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")

    session = get_session_or_404(session_id)
    ensure_retrieval(session, top_k_per_query=top_k_per_query)

    return {
        "session_id": session_id,
        "total_queries": len(session["multi_queries"]),
        "top_k_per_query": top_k_per_query,
        "retrieval_results": session["retrieval_results"],
        "total_chunks": len(session["retrieval_flat_chunks"])
    }


@app.post("/rerank")
def rerank(data: Dict[str, Any]):
    session_id = data.get("session_id")
    top_n = int(data.get("top_n", DEFAULT_RERANK_TOP_N))

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")

    session = get_session_or_404(session_id)
    ensure_rerank(session, top_n=top_n)

    return {
        "session_id": session_id,
        "final_user_query": session["final_user_query"],
        "reranked_chunks": session["reranked_chunks"],
        "top_chunks": session["top_chunks"]
    }


@app.post("/answer/generate")
def answer_generate(data: Dict[str, Any]):
    session_id = data.get("session_id")

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")

    session = get_session_or_404(session_id)

    if not session["top_chunks"]:
        raise HTTPException(status_code=400, detail="rerank not done yet")

    request_id = start_final_answer_job(session)

    return {
        "session_id": session_id,
        "request_id": request_id,
        "status": "processing",
        "polling_interval_sec": 5,
        "result_url": f"/answer/result/{request_id}"
    }


@app.get("/answer/result/{request_id}")
def answer_result(request_id: str):
    data = requests_store.get(request_id)

    if not data:
        return {"status": "not_found"}

    if data["status"] == "processing":
        return {
            "status": "processing",
            "request_id": request_id,
            "polling_interval_sec": 5
        }

    if data["status"] == "error":
        return {
            "status": "error",
            "result": data["result"]
        }

    res = data["result"]
    res["source"] = "fallback_get"

    return {"status": "done", "result": res}


@app.get("/answer/stream/{request_id}")
def answer_stream_disabled(request_id: str):
    return {
        "status": "disabled",
        "message": "SSE disabled. Use GET /answer/result/{request_id} every 5 seconds."
    }


@app.post("/pipeline/run")
def pipeline_run(data: Dict[str, Any]):
    session_id = data.get("session_id")
    top_k_per_query = int(data.get("top_k_per_query", DEFAULT_TOP_K))
    top_n = int(data.get("top_n", DEFAULT_RERANK_TOP_N))

    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required")

    session = get_session_or_404(session_id)

    if not session["final_user_query"]:
        raise HTTPException(status_code=400, detail="final_user_query not prepared yet")

    if not session["multi_queries"]:
        result = generate_multi_queries(session["final_user_query"], count=DEFAULT_MULTI_QUERY_COUNT)
        session["multi_queries"] = result["data"].get("multi_queries", [])
        update_step_metrics(session, "query_multi", {
            "usage": result["usage"],
            "cost": result["cost"],
            "timing": result["timing"],
            "meta": {
                "generated_queries": len(session["multi_queries"])
            }
        })

    ensure_retrieval(session, top_k_per_query=top_k_per_query)
    ensure_rerank(session, top_n=top_n)

    request_id = start_final_answer_job(session)

    return {
        "session_id": session_id,
        "final_user_query": session["final_user_query"],
        "multi_queries": session["multi_queries"],
        "retrieval_results": session["retrieval_results"],
        "reranked_chunks": session["reranked_chunks"],
        "top_chunks": session["top_chunks"],
        "answer_request": {
            "request_id": request_id,
            "polling_interval_sec": 5,
            "result_url": f"/answer/result/{request_id}"
        },
        "status": "processing_final_answer"
    }
'''

with open("/content/backend_app.py", "w", encoding="utf-8") as f:
    f.write(backend_code)

print("✅ /content/backend_app.py created")

# ---------- 4) Start server ----------
import subprocess, socket, re

PORT = 8000

subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

uvicorn = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend_app:app", "--host", "0.0.0.0", "--port", str(PORT), "--app-dir", "/content"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def wait_port(port):
    for _ in range(60):
        s = socket.socket()
        try:
            s.connect(("127.0.0.1", port))
            s.close()
            return True
        except:
            time.sleep(1)
    return False

if not wait_port(PORT):
    print(uvicorn.stdout.read())
    raise RuntimeError("Server not started")

# ---------- 5) Cloudflare ----------
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None
for _ in range(60):
    line = proc.stdout.readline()
    m = re.search(r"(https://.*trycloudflare.com)", line)
    if m:
        url = m.group(1)
        break

if not url:
    raise RuntimeError("Tunnel failed")

print("="*60)
print("🚀 BACKEND READY")
print("="*60)
print("Public URL:", url)
print("\nEndpoints:")
print(url + "/health")
print(url + "/session/start")
print(url + "/session/{session_id}")
print(url + "/question/clarify")
print(url + "/question/clarify/answer")
print(url + "/query/multi")
print(url + "/retrieve")
print(url + "/rerank")
print(url + "/answer/generate")
print(url + "/answer/result/{request_id}")
print(url + "/answer/stream/{request_id}   # disabled, returns instruction JSON")
print(url + "/pipeline/run")
print("="*60)

^C
^C
✅ API key loaded from Colab Secrets
✅ /content/backend_app.py created
🚀 BACKEND READY
Public URL: https://pit-shorter-floral-question.trycloudflare.com

Endpoints:
https://pit-shorter-floral-question.trycloudflare.com/health
https://pit-shorter-floral-question.trycloudflare.com/session/start
https://pit-shorter-floral-question.trycloudflare.com/session/{session_id}
https://pit-shorter-floral-question.trycloudflare.com/question/clarify
https://pit-shorter-floral-question.trycloudflare.com/question/clarify/answer
https://pit-shorter-floral-question.trycloudflare.com/query/multi
https://pit-shorter-floral-question.trycloudflare.com/retrieve
https://pit-shorter-floral-question.trycloudflare.com/rerank
https://pit-shorter-floral-question.trycloudflare.com/answer/generate
https://pit-shorter-floral-question.trycloudflare.com/answer/result/{request_id}
https://pit-shorter-floral-question.trycloudflare.com/answer/stream/{request_id}   # disabled, returns instruction JSON
https://pit-shor